# Loadgen spoke **D**

One of five spokes. Fires `calls` concurrent HTTPS requests at the Azure Function using a `ThreadPoolExecutor`, captures per-call latency, returns aggregated stats via `notebook.exit(json...)`.

Can run standalone for debugging — defaults below produce a tiny test run.

In [ ]:
# === Parameters (tagged so parent overwrites at runtime) ===
# IMPORTANT: Fabric notebook.run / runMultiple accept ONLY flat
# primitive values in `arguments` -- nested dicts/lists silently
# kill the spawned Spark session. Complex payloads MUST be passed
# as a JSON string (`payload_json`) and parsed in this notebook.
function_url    = "https://my-fn-app.azurewebsites.net/api/HelloWorld"
function_key    = "REPLACE_WITH_FUNCTION_KEY"
calls           = 10              # standalone test default; orchestrator passes 1000
concurrency     = 5               # standalone test default; orchestrator passes 50
request_timeout = 30
shard_id        = "D"
payload_json    = '{"name": "spoke-D-standalone"}'

In [ ]:
# === Fire `calls` concurrent requests ===
import json, time, requests, statistics as stats
# Parse JSON-string payload from parent (Fabric arguments must be flat primitives)
try:
    payload = json.loads(payload_json) if isinstance(payload_json, str) else (payload_json or {})
except (TypeError, ValueError):
    payload = {}
from concurrent.futures import ThreadPoolExecutor, as_completed

session = requests.Session()
session.headers.update({
    "Content-Type":    "application/json",
    "x-functions-key": function_key,
})

def _one_call(i: int):
    t0 = time.time()
    try:
        body = {**payload, "shard": shard_id, "call_index": i}
        r = session.post(function_url, data=json.dumps(body), timeout=request_timeout)
        ms = (time.time() - t0) * 1000
        return (r.status_code, ms, None)
    except Exception as e:
        ms = (time.time() - t0) * 1000
        return (None, ms, f"{type(e).__name__}: {e}")

latencies_ok = []
latencies_all = []
ok_count = 0
fail_count = 0
fail_samples = []

wall_t0 = time.time()
with ThreadPoolExecutor(max_workers=concurrency) as ex:
    futures = [ex.submit(_one_call, i) for i in range(calls)]
    for fut in as_completed(futures):
        status, ms, err = fut.result()
        latencies_all.append(ms)
        if err is None and status and 200 <= status < 300:
            ok_count += 1
            latencies_ok.append(ms)
        else:
            fail_count += 1
            if len(fail_samples) < 5:
                fail_samples.append({"status": status, "error": err})
wall_s = time.time() - wall_t0

def _pct(xs, p):
    if not xs: return 0
    if len(xs) < 2: return xs[0]
    q = stats.quantiles(xs, n=100)
    return q[min(p-1, 98)]

result = {
    "shard_id":       shard_id,
    "count_ok":       ok_count,
    "count_fail":     fail_count,
    "wall_seconds":   round(wall_s, 2),
    "throughput_rps": round(calls / wall_s, 2) if wall_s else 0,
    "p50_ms":         round(stats.median(latencies_ok)) if latencies_ok else 0,
    "p95_ms":         round(_pct(latencies_ok, 95)),
    "p99_ms":         round(_pct(latencies_ok, 99)),
    "max_ms":         round(max(latencies_ok)) if latencies_ok else 0,
    "fail_samples":   fail_samples,
    # Truncate raw latencies to keep exit payload < ~5MB
    "latencies_ms":   [round(x, 1) for x in latencies_ok[:5000]],
}

print(json.dumps({k: v for k, v in result.items() if k != "latencies_ms"}, indent=2))
print(f"(returning {len(result['latencies_ms'])} latency samples to parent)")

In [ ]:
# === Return to parent ===
import json
try:
    from notebookutils import notebook
except Exception:
    from mssparkutils import notebook
notebook.exit(json.dumps(result))